# 02 - Prétraitement et EDA (Phase 3)
**Objectif :** explorer les ventes, justifier la définition du churn (distribution des délais inter-achats), mesurer la concentration du CA (Pareto), la rétention par cohorte et le profil B2B.

> Prérequis : `python run_pipeline.py` (ou le notebook 01) pour produire `data/processed/transactions_clean`.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config, resolve, save_table, load_table
cfg = load_config()
FIG = resolve(cfg, "figures_dir")
OUT = resolve(cfg, "outputs_dir")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": .3})

df = load_table("transactions_clean", cfg)
sales = df[~df["IsReturn"]]
print(f"{len(df):,} lignes | période {df['InvoiceDate'].min():%Y-%m-%d} -> {df['InvoiceDate'].max():%Y-%m-%d}")

## 1. Évolution mensuelle des ventes (CA net = ventes - retours)

In [ ]:
m = df.groupby("YearMonth").agg(ca_net=("Revenue", "sum"), commandes=("Invoice", lambda s: s[~df.loc[s.index, "IsReturn"]].nunique()),
                                 clients=("CustomerID", "nunique"))
fig, ax = plt.subplots(); m["ca_net"].plot(kind="bar", ax=ax, color="#2b6cb0"); ax.set_title("CA net mensuel (GBP)")
plt.xticks(rotation=60); plt.tight_layout(); plt.savefig(FIG / "ca_mensuel.png", dpi=150); plt.show()
m.round(0)

**Attention :** décembre 2011 est incomplet (données jusqu'au 9 décembre). Ne pas conclure à une chute.

## 2. Saisonnalité : mois, jour de la semaine, heure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
sales.groupby("Month")["Revenue"].sum().plot(kind="bar", ax=axes[0], title="CA par mois de l'année")
sales.groupby("DayOfWeek")["Invoice"].nunique().plot(kind="bar", ax=axes[1], title="Commandes par jour (0 = lundi)")
sales.groupby("Hour")["Invoice"].nunique().plot(kind="bar", ax=axes[2], title="Commandes par heure")
plt.tight_layout(); plt.savefig(FIG / "saisonnalite.png", dpi=150); plt.show()

## 3. Pays

In [ ]:
c = sales.groupby("Country")["Revenue"].sum().sort_values(ascending=False)
print((100 * c / c.sum()).head(10).round(2))
uk = c.get("United Kingdom", 0) / c.sum()
print(f"\nRoyaume-Uni = {100*uk:.1f} % du CA -> pays exclu du clustering, utilisé pour profiler les segments")

## 4. Produits populaires

In [ ]:
top_q = sales.groupby("Description")["Quantity"].sum().nlargest(10)
top_r = sales.groupby("Description")["Revenue"].sum().nlargest(10)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
top_q[::-1].plot(kind="barh", ax=axes[0], title="Top 10 par quantité"); top_r[::-1].plot(kind="barh", ax=axes[1], title="Top 10 par CA")
plt.tight_layout(); plt.savefig(FIG / "top_produits.png", dpi=150); plt.show()
print("Répartition du CA par catégorie dérivée (approximative) :")
print((100 * sales.groupby("Category")["Revenue"].sum() / sales["Revenue"].sum()).sort_values(ascending=False).round(1))

## 5. Distribution du panier (CA par facture)

In [ ]:
basket = sales.groupby("Invoice")["Revenue"].sum()
print(basket.describe().round(1))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
basket.clip(upper=basket.quantile(.99)).plot(kind="hist", bins=60, ax=ax[0], title="Panier (tronqué au 99e percentile)")
np.log10(basket[basket > 0]).plot(kind="hist", bins=60, ax=ax[1], title="log10(panier) : distribution très asymétrique")
plt.tight_layout(); plt.savefig(FIG / "paniers.png", dpi=150); plt.show()

## 6. Courbe de Pareto : concentration du CA par client
Mesure combien de clients font l'essentiel du CA (et le poids des grossistes).

In [ ]:
cust = df[df["HasCustomer"] & ~df["IsReturn"]].groupby("CustomerID")["Revenue"].sum().sort_values(ascending=False)
cum = cust.cumsum() / cust.sum(); share_x = np.arange(1, len(cust) + 1) / len(cust)
fig, ax = plt.subplots(); ax.plot(share_x * 100, cum.values * 100); ax.axhline(80, color="grey", ls="--")
ax.set_xlabel("% des clients (du plus gros au plus petit)"); ax.set_ylabel("% du CA cumulé"); ax.set_title("Courbe de Pareto")
plt.savefig(FIG / "pareto.png", dpi=150); plt.show()
top20 = cum.iloc[int(0.2 * len(cust)) - 1]; top1 = cust.head(max(1, int(0.01 * len(cust)))).sum() / cust.sum()
print(f"Les 20 % de clients les plus gros font {100*top20:.1f} % du CA ; le top 1 % fait {100*top1:.1f} %")

## 7. Délais inter-achats : justification de l'horizon de churn
**Question :** au bout de combien de jours sans achat un client est-il « probablement perdu » ?
On regarde le délai entre deux achats consécutifs (jours distincts) de chaque client. Si 80 % des clients qui rachètent le font en moins de *X* jours, un silence de *X* jours est un signal raisonnable.

In [ ]:
days = (sales[sales["HasCustomer"]][["CustomerID", "InvoiceDay"]].drop_duplicates().sort_values(["CustomerID", "InvoiceDay"]))
gaps = days.groupby("CustomerID")["InvoiceDay"].diff().dt.days.dropna()
pct = gaps.quantile([.5, .7, .8, .9, .95]).round(0)
print("Percentiles des délais entre deux achats (jours) :"); print(pct)
fig, ax = plt.subplots(); gaps.clip(upper=365).plot(kind="hist", bins=73, ax=ax)
for p, col in [(.8, "red"), (.9, "orange")]:
    ax.axvline(gaps.quantile(p), color=col, ls="--", label=f"p{int(p*100)} = {gaps.quantile(p):.0f} j")
ax.axvline(cfg["churn"]["horizon_days"], color="green", label=f"horizon choisi = {cfg['churn']['horizon_days']} j")
ax.legend(); ax.set_title("Délais entre achats consécutifs (tronqué à 365 j)"); ax.set_xlabel("jours")
plt.savefig(FIG / "delais_inter_achats.png", dpi=150); plt.show()
p80, h = gaps.quantile(.8), cfg["churn"]["horizon_days"]
print(f"\np80 des délais = {p80:.0f} j | p90 = {gaps.quantile(.9):.0f} j | horizon configuré = {h} j")
print("-> Si l'horizon est très éloigné du p80, argumente le choix (ou ajuste churn.horizon_days dans config.yaml) puis reporte-le dans le cadrage.")
print(f"Sensibilité testée au notebook 03 : {cfg['churn']['sensitivity_horizons']}")

## 8. Cohortes de rétention (mois de première commande)

In [ ]:
first = sales[sales["HasCustomer"]].groupby("CustomerID")["InvoiceDay"].min().dt.to_period("M").rename("cohort")
act = sales[sales["HasCustomer"]].merge(first, on="CustomerID")
act["period"] = act["InvoiceDay"].dt.to_period("M")
act["age"] = (act["period"] - act["cohort"]).apply(lambda x: x.n)
coh = act.groupby(["cohort", "age"])["CustomerID"].nunique().unstack(fill_value=0)
ret = coh.div(coh[0], axis=0).round(3)
fig, ax = plt.subplots(figsize=(12, 7)); im = ax.imshow(ret.iloc[:, :13].values, cmap="Blues", vmin=0, vmax=0.6, aspect="auto")
ax.set_yticks(range(len(ret))); ax.set_yticklabels(ret.index.astype(str)); ax.set_xlabel("mois depuis la première commande")
ax.set_title("Rétention par cohorte (part des clients actifs)"); plt.colorbar(im); plt.savefig(FIG / "cohortes.png", dpi=150); plt.show()
print("Rétention moyenne à M+1, M+3, M+6, M+12 :"); print(ret[[1, 3, 6, 12]].mean().round(3) if ret.shape[1] > 12 else ret.mean().round(3).head(13))

## 9. Retours

In [ ]:
ret_df = df[df["IsReturn"]]
print(f"Lignes de retour : {len(ret_df):,} ({100*len(ret_df)/len(df):.2f} %) | montant retourné : {ret_df['Revenue'].sum():,.0f} GBP "
      f"({100*abs(ret_df['Revenue'].sum())/sales['Revenue'].sum():.2f} % du CA brut)")
print("Part des retours sans CustomerID :", f"{100*(~ret_df['HasCustomer']).mean():.1f} %")

## 10. Profil B2B / B2C : justifier le seuil `is_wholesale`
Le seuil est **fixe** dans `config.yaml` (jamais calculé sur toutes les données, pour éviter toute fuite). Ce graphique sert à le justifier.

In [ ]:
c = sales[sales["HasCustomer"]].groupby("CustomerID").agg(qty=("Quantity", "sum"), lines=("StockCode", "size"), rev=("Revenue", "sum"), n_inv=("Invoice", "nunique"))
c["avg_qty_per_line"] = c["qty"] / c["lines"]
thr = cfg["features"]["wholesale_avg_qty_per_line"]
fig, ax = plt.subplots(); np.log10(c["avg_qty_per_line"]).plot(kind="hist", bins=60, ax=ax)
ax.axvline(np.log10(thr), color="red", ls="--", label=f"seuil = {thr}"); ax.legend(); ax.set_title("log10 de la quantité moyenne par ligne, par client")
plt.savefig(FIG / "profil_b2b.png", dpi=150); plt.show()
w = c["avg_qty_per_line"] >= thr
print(f"Clients 'grossistes' : {100*w.mean():.1f} % des clients, {100*c.loc[w,'rev'].sum()/c['rev'].sum():.1f} % du CA")

## 11. Conclusions de l'EDA (à compléter)
- Saisonnalité : ...
- Concentration (Pareto) : ...
- Horizon de churn justifié par p80 = ... jours : ...
- Rétention : ...
- Part du B2B : ...